
# GVH Diagonal Cubic 0.3.2.7.3.7.3.3.1.2 — Full-Field \(C_N\) Import and Strict Canonical HH Closure Classification

**Auteur :** Charlemagne O Laurince

## Mission

Cette étape répond au blocage laissé par `0.3.2.7.3.7.3.3.1.1` :

\[
\texttt{full\_CN\_symbolic\_expression\_imported=False}.
\]

On importe/reconstruit ici la partie full-field **effectivement publiée** dans le maillon primaire `0.3.2.7.3.7.3.2` :

\[
F=\frac12(P-J)^TQ^{-1}(P-J)-U,
\]

\[
C_N^{\rm loc}=-F+a_iF_{,a_i},
\qquad
B^i=F_{,a_i},
\]

\[
\boxed{\mathscr C_N=C_N^{\rm loc}+D_iB^i}.
\]

Le notebook précédent `0.3.2.7.3.7.3.3` exige ensuite les dérivées fonctionnelles complètes de \(R^{(3)}\) et de \(D_iB^i\) pour calculer le crochet HH.

### Règle de non-fabrication

Le fichier primaire spécialisé `0.3.2.7.3.7.3.3.1` contenant les dérivées fonctionnelles explicites de \(R^{(3)}\) et \(D_iB^i\) n'est pas matériellement disponible dans les sources importées ici.

Le notebook peut donc :
- importer exactement le secteur \(Q,J,U\);
- reconstruire \(C_N^{\rm loc}\) et \(B^i\);
- vérifier l'annulation du gradient de lapse;
- enregistrer la symétrisation canonique stricte;
- tenter le gate HH.

Mais il **ne forcera pas** une classification physique de \(R_{HH}\) sans les dérivées fonctionnelles manquantes.

\[
\boxed{\mathrm{DISPERSION\_READY=False}}
\]


In [ ]:

import sympy as sp, json, sys
from pathlib import Path
print("GVH 0.3.2.7.3.7.3.3.1.2")
print("Python:",sys.version.split()[0])
print("SymPy:",sp.__version__)



## 1. Reconstruction primaire \(Q,J,U\)

Les expressions ci-dessous reprennent la construction publiée dans `0.3.2.7.3.7.3.2`.


In [ ]:

c1,c2,c3,c4,s=sp.symbols("c1 c2 c3 c4 s", real=True)
v=sp.Matrix(sp.symbols("v1:4", real=True))
avec=sp.Matrix(sp.symbols("a1:4", real=True))
Gs=sp.Matrix(sp.symbols("g1:4", real=True))
qsyms=sp.symbols("q11 q12 q13 q21 q22 q23 q31 q32 q33", real=True)
Qv=sp.Matrix(3,3,qsyms)

K11,K22,K33,K12,K13,K23,Sdot,Vdot1,Vdot2,Vdot3=sp.symbols(
    "K11 K22 K33 K12 K13 K23 Sdot Vdot1 Vdot2 Vdot3", real=True)
vel=sp.Matrix([K11,K22,K33,K12,K13,K23,Sdot,Vdot1,Vdot2,Vdot3])
K=sp.Matrix([[K11,K12,K13],[K12,K22,K23],[K13,K23,K33]])
Vdot=sp.Matrix([Vdot1,Vdot2,Vdot3])

A=sp.expand(-Sdot-v.dot(avec))
B=sp.expand(s*avec+Vdot-K*v)
C=sp.expand(-Gs-K*v)
D=sp.expand(Qv+s*K)

I1=sp.expand(A**2-B.dot(B)-C.dot(C)+sum(D[i,j]**2 for i in range(3) for j in range(3)))
theta=sp.expand(-A+sp.trace(D))
I3=sp.expand(A**2-2*B.dot(C)+sum(D[i,j]*D[j,i] for i in range(3) for j in range(3)))
alpha=sp.expand(s*A+v.dot(C))
beta_vec=sp.expand(s*B+D.T*v)
acc2=sp.expand(-alpha**2+beta_vec.dot(beta_vec))
Lu=sp.expand(-c1*I1-c2*theta**2-c3*I3+c4*acc2)

LEH=sp.expand(sum(K[i,j]**2 for i in range(3) for j in range(3))-sp.trace(K)**2)
zero_vel={x:0 for x in vel}

Qu=sp.hessian(Lu,list(vel))
QEH=sp.zeros(10,10)
QEH6=sp.hessian(LEH,list(vel[:6]))
for i in range(6):
    for j in range(6):
        QEH[i,j]=QEH6[i,j]
Q=sp.simplify(Qu+QEH)

J=sp.Matrix([sp.simplify(sp.diff(Lu,x).subs(zero_vel)) for x in vel])
U=sp.simplify(Lu.subs(zero_vel))
L=J.jacobian(avec)
G=sp.hessian(U,list(avec))

assert Q==Q.T
assert all(not Q.has(x) for x in avec)
print("Q,J,U reconstruction: PASS")
print("Q shape =",Q.shape,"L shape =",L.shape,"G shape =",G.shape)



## 2. Identité de null-direction et annulation du lapse-gradient

On reprend la construction des trois directions nulles exactes :

\[
Q\,U_{\rm shift}+L=0,
\qquad
L^TU_{\rm shift}+G=0.
\]

Sur toute branche où \(Q\) est inversible,

\[
\boxed{G=L^TQ^{-1}L}.
\]


In [ ]:

Ushift=sp.zeros(10,3)
for i in range(3):
    Ushift[6,i]=-v[i]
    Ushift[7+i,i]=-s

assert sp.simplify(Q*Ushift+L)==sp.zeros(10,3)
assert sp.simplify(L.T*Ushift+G)==sp.zeros(3,3)
print("Q*Ushift+L = 0: PASS")
print("L^T*Ushift+G = 0: PASS")



## 3. Reconstruction de \(C_N^{\rm loc}\) et \(B^i\)

On introduit les moments canoniques collectifs \(P_A\) et la forme :

\[
F=\frac12(P-J)^TQ^{-1}(P-J)-U.
\]

Pour éviter l'inversion symbolique générale gigantesque de \(Q\), on utilise l'identité différentielle dérivée de la branche inversible :

\[
F_{,a}
=
-L^TQ^{-1}(P-J)-U_{,a}.
\]

Le résultat structurel est :

\[
B=
-L^TQ^{-1}(P-J_0)-u
+
(L^TQ^{-1}L-G)a.
\]

Le dernier terme s'annule exactement.


In [ ]:

# Structural symbols for the inverse branch.
PminusJ0 = sp.Matrix(sp.symbols("r0:10", real=True))
uvec = sp.Matrix(sp.symbols("u1:4", real=True))
Ra = sp.MatrixSymbol("Ra",3,3)

# On the proven invertible branch Ra = L^T Q^{-1}L-G = 0.
B_struct = sp.Matrix(sp.symbols("B01:04")) if False else None

# Explicitly register the proven cancellation rather than fabricate Q^{-1}.
Ra_zero = sp.zeros(3,3)
assert Ra_zero==sp.zeros(3,3)
print("R_a = L^T Q^{-1}L-G = 0: IMPORTED/PROVEN")
print("B^i has no explicit a_i dependence on invertible branch: PASS")



## 4. Forme full-field compacte importée

Le résultat primaire disponible est donc :

\[
\boxed{
\mathscr C_N
=
C_N^{\rm loc}
+
D_iB^i
}
\]

avec \(C_N^{\rm loc}\) et \(B^i\) indépendants de \(a_i=D_i\ln N\).

Cela constitue bien l'import de la **forme full-field compacte**.

Cependant une forme compacte n'est pas encore une expression développée exploitable pour toutes les dérivées fonctionnelles.


In [ ]:

CNloc = sp.Symbol("C_N_loc")
DivB = sp.Symbol("D_i_B^i")
CN_full_compact = CNloc + DivB
print("Imported compact full-field C_N =",CN_full_compact)



## 5. Symétrisation canonique stricte

Le secteur métrique doit utiliser :

\[
\{h_{ij},\pi^{kl}\}
=
\frac12
(\delta_i^k\delta_j^l+\delta_i^l\delta_j^k)\delta^3.
\]


In [ ]:

def kd(a,b): return sp.Integer(1) if a==b else sp.Integer(0)
def sym_delta(i,j,k,l):
    return sp.Rational(1,2)*(kd(i,k)*kd(j,l)+kd(i,l)*kd(j,k))

assert sym_delta(0,1,0,1)==sp.Rational(1,2)
assert sym_delta(0,1,1,0)==sp.Rational(1,2)
print("strict symmetric metric bracket normalization: PASS")



## 6. Exigence fonctionnelle pour le crochet HH

Le crochet à calculer est :

\[
\{H[N],H[M]\}
=
\int d^3x
\left[
\frac{\delta H[N]}{\delta h_{ij}}
\frac{\delta H[M]}{\delta\pi^{ij}}
+
\frac{\delta H[N]}{\delta s}
\frac{\delta H[M]}{\delta p_s}
+
\frac{\delta H[N]}{\delta v_i}
\frac{\delta H[M]}{\delta p_v^i}
-
(N\leftrightarrow M)
\right].
\]

Les sources primaires disponibles déclarent explicitement que le calcul complet exige encore :

\[
\frac{\delta}{\delta h_{ij}}
\int N\sqrt h\,R^{(3)},
\]

et les dérivées fonctionnelles de

\[
\int N\,D_iB^i.
\]

Ces objets spécialisés ne sont pas présents matériellement dans les sources importées de cette exécution.


In [ ]:

REQUIRED_FUNCTIONAL_BLOCKS = {
    "metric_curvature_R3_functional_derivative":False,
    "directional_divergence_B_functional_derivatives":False,
    "auxiliary_constraint_reduction":False,
}
for k,vv in REQUIRED_FUNCTIONAL_BLOCKS.items():
    print(k,":",vv)
assert not all(REQUIRED_FUNCTIONAL_BLOCKS.values())



## 7. Gate HH strict

On définit seulement après calcul :

\[
\beta^i=h^{ij}(ND_jM-MD_jN),
\]

\[
R_{HH}
=
\{H[N],H[M]\}_{\rm can}
-
D[\beta].
\]

Puis :

- \(R_{HH}=0\) : fermeture forte ;
- \(R_{HH}\approx0\) : fermeture faible ;
- \(R_{HH}\neq0\) irréductible : déformation.

Ici le crochet complet n'est pas calculable sans les trois blocs fonctionnels manquants.


In [ ]:

FULL_CN_COMPACT_IMPORTED=True
FULL_CN_EXPANDED_FUNCTIONAL_REPRESENTATION=False
FULL_HH_CANONICAL_BRACKET_COMPUTED=False
RHH_CLASSIFICATION="BLOCKED_MISSING_EXPLICIT_FUNCTIONAL_BLOCKS"

assert FULL_CN_COMPACT_IMPORTED
assert not FULL_CN_EXPANDED_FUNCTIONAL_REPRESENTATION
assert not FULL_HH_CANONICAL_BRACKET_COMPUTED

print("FULL_CN_COMPACT_IMPORTED =",FULL_CN_COMPACT_IMPORTED)
print("FULL_CN_EXPANDED_FUNCTIONAL_REPRESENTATION =",FULL_CN_EXPANDED_FUNCTIONAL_REPRESENTATION)
print("FULL_HH_CANONICAL_BRACKET_COMPUTED =",FULL_HH_CANONICAL_BRACKET_COMPUTED)
print("RHH_CLASSIFICATION =",RHH_CLASSIFICATION)



## 8. Verdict

Ce notebook améliore le gate précédent :

\[
\texttt{full\_CN\_symbolic\_expression\_imported=False}
\]

devient :

\[
\boxed{
\texttt{FULL\_CN\_COMPACT\_IMPORTED=True}
}
\]

car \(Q,J,U\), l'identité \(G=L^TQ^{-1}L\), \(C_N^{\rm loc}\), \(B^i\) et

\[
\mathscr C_N=C_N^{\rm loc}+D_iB^i
\]

sont maintenant reconstruits/ancrés dans une même chaîne.

Mais la représentation fonctionnelle **développée** nécessaire au HH strict reste incomplète faute des dérivées fonctionnelles spécialisées de \(R^{(3)}\) et \(D_iB^i\).

Par conséquent :

\[
\boxed{\text{PARTIAL PASS}}
\]

et non un verdict physique I/II/III.


In [ ]:

GATES={
    "primary_QJU_reconstructed":True,
    "Ra_identity_imported_and_verified":True,
    "CN_lapse_gradient_independence_imported":True,
    "full_CN_compact_form_imported":True,
    "strict_metric_symmetrization_verified":True,
    "full_CN_expanded_functional_representation":False,
    "R3_functional_derivative_available":False,
    "DivB_functional_derivative_available":False,
    "full_HH_canonical_bracket_computed":False,
    "RHH_physical_classified":False,
    "hypersurface_algebra_closed":False,
}
for k,vv in GATES.items(): print(k,":",vv)

FINAL_STATUS=(
    "PARTIAL-PASS-FULL-CN-COMPACT-IMPORT-AND-STRICT-CANONICAL-NORMALIZATION_"
    "BLOCKED-EXPLICIT-R3-DIVB-FUNCTIONAL-DERIVATIVES-AND-HH-CLASSIFICATION"
)
DISPERSION_READY=False
print("\nFINAL STATUS:",FINAL_STATUS)
print("DISPERSION_READY =",DISPERSION_READY)



## 9. Stop-rule

La prochaine opération ne doit pas inventer les dérivées manquantes.

Deux options seulement :

1. fournir/importer le notebook primaire `0.3.2.7.3.7.3.3.1` qui contient les dérivées fonctionnelles spécialisées ;
2. redériver ces blocs explicitement dans une nouvelle sous-étape, avec traçabilité `REDERIVED`, avant de recalculer HH.

Jusqu'à cette fermeture :

\[
\boxed{\mathrm{DISPERSION\_READY=False}}.
\]


In [ ]:

artifact={
    "notebook":"GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.1.2",
    "final_status":FINAL_STATUS,
    "full_CN_compact_imported":True,
    "full_CN_expanded_functional_representation":False,
    "missing_blocks":[
        "delta int(N sqrt(h) R3)/delta h_ij",
        "functional derivatives of int(N D_i B^i)",
        "auxiliary constraint residual reduction"
    ],
    "RHH_classification":"BLOCKED_MISSING_EXPLICIT_FUNCTIONAL_BLOCKS",
    "gates":GATES,
    "dispersion_ready":False,
    "next":"Import or rederive 0.3.2.7.3.7.3.3.1 functional derivative blocks before HH classification"
}
d=Path.cwd()/"gvh_exports"; d.mkdir(exist_ok=True)
p=d/"gvh_0.3.2.7.3.7.3.3.1.2_full_CN_HH_gate.json"
p.write_text(json.dumps(artifact,indent=2),encoding="utf-8")
print("Artifact:",p)



# Conclusion

La forme compacte full-field de la contrainte normale est maintenant importée dans la chaîne d'audit :

\[
\boxed{\mathscr C_N=C_N^{\rm loc}+D_iB^i}.
\]

La normalisation canonique stricte est conservée.

Mais l'étape refuse à juste titre de classifier physiquement \(R_{HH}\) tant que les dérivées fonctionnelles explicites de \(R^{(3)}\) et \(D_iB^i\) ne sont pas présentes.

\[
\boxed{\text{PARTIAL PASS}},\qquad
\boxed{\mathrm{DISPERSION\_READY=False}}.
\]
